# H&M Transaction Data: Product Recommendations 01

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import sys

import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# data processing classes
from src.customer_features import CustomerFeatureEngineer
from src.product_features import ProductFeatureEngineer
from src.recommendation_training import RecommendationTrainingBuilder

E0000 00:00:1771647857.437919 3923909 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1771647857.437936 3923909 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1771647857.437938 3923909 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1771647857.437939 3923909 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1771647857.437940 3923909 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


## Data loading

In [2]:
data_path = Path("../data")
customers = pd.read_csv(data_path / 'customer_hm_cleaned.csv')
transactions = pd.read_csv(data_path / 'transactions_hm_cleaned.csv')
articles = pd.read_csv(data_path / 'articles_hm_cleaned.csv')

In [3]:
print("Cleaned Rows of Data:")
print(f"Articles: {len(articles):,}")
print(f"Customers: {len(customers):,}")
print(f"Transactions: {len(transactions):,}")

Cleaned Rows of Data:
Articles: 105,542
Customers: 1,048,575
Transactions: 1,040,101


In [4]:
# Sample data for faster execution
TRANSACTIONS_SAMPLE_SIZE = 500000
transactions_sample = transactions.sample(n=TRANSACTIONS_SAMPLE_SIZE, random_state=67)

CUSTOMER_SAMPLE_SIZE = 500000
# sample_customers = transactions_sample['customer_id'].unique()
customers_sample = customers.sample(n=CUSTOMER_SAMPLE_SIZE, random_state=67)

transactions_df = transactions_sample
customers_df = customers_sample
articles_df = articles

In [5]:
print(f"Customers: using {len(customers_df):,} out of {len(customers):,} available")
print(f"Transactions: using {len(transactions_df):,} out of {len(transactions):,} available")
print(f"Articles: using {len(articles_df):,} out of {len(articles):,} available")

Customers: using 500,000 out of 1,048,575 available
Transactions: using 500,000 out of 1,040,101 available
Articles: using 105,542 out of 105,542 available


## Train, Validation Data Generation

In [6]:
def build_product_recommendation_data(as_of_date, prediction_start_date, prediction_end_date):
    print(f"Data as of date: {as_of_date}")

    cfe = CustomerFeatureEngineer(customers_df=customers_df, transactions_df=transactions_df)
    customer_features = cfe.calculate_all_features(articles_df=articles_df, as_of_date=as_of_date)
    print(f"Customer rows: {len(customer_features):,}")

    pfe = ProductFeatureEngineer(articles_df=articles_df, transactions_df=transactions_df)
    product_features = pfe.calculate_all_features(as_of_date=as_of_date)
    print(f"Product rows: {len(product_features):,}")


    print(f"Prediction period: {prediction_start_date} to {prediction_end_date}")
    builder = RecommendationTrainingBuilder(transactions_df=transactions_df,
                                                customer_features_df=customer_features,
                                                product_features_df=product_features)
    data = builder.build_dataset(prediction_start=prediction_start_date,
                                            prediction_end=prediction_end_date,
                                            negative_ratio=5,
                                            random_state=67)

    print(f"Total rows: {len(data):,}")
    print(f"- Positives: {(data['purchased'] == 1).sum():,}")
    print(f"- Negatives: {(data['purchased'] == 0).sum():,}")
    return data

In [ ]:
print("=========TRAINING DATA=========")
train_as_of_date = "2019-09-30"
train_prediction_start_date = "2019-10-01"
train_prediction_end_date = "2019-10-30"
train_data =  build_product_recommendation_data(train_as_of_date, train_prediction_start_date, train_prediction_end_date)

=========TRAINING DATA=========
Data as of date: 2019-09-30
Customer rows: 252,238
Product rows: 37,553
Prediction period: 2019-10-01 to 2019-10-30


In [ ]:
print("\n=========VALIDATION DATA=========")
val_as_of_date = "2019-10-31"
val_prediction_start = "2019-11-01"
val_prediction_end = "2019-11-30"
val_data =  build_product_recommendation_data(val_as_of_date, val_prediction_start, val_prediction_end)


=========VALIDATION DATA=========
Data as of date: 2019-10-31
Customer rows: 40,243
Product rows: 17,066
Prediction period: 2019-11-01 to 2019-11-30
Total rows: 1,260
- Positives: 210
- Negatives: 1,050


In [ ]:
print("\n=========TEST DATA=========")
test_as_of_date = "2019-11-30"
test_prediction_start = "2019-12-01"
test_prediction_end = "2019-12-31"
test_data =  build_product_recommendation_data(test_as_of_date, test_prediction_start, test_prediction_end)


=========TEST DATA=========
Data as of date: 2019-11-30
Customer rows: 43,487
Product rows: 18,160
Prediction period: 2019-12-01 to 2019-12-31
Total rows: 1,158
- Positives: 193
- Negatives: 965


In [ ]:
feature_cols = [col for col in train_data.columns if col not in ['customer_id', 'article_id', 'purchased']]

print(f"Feature Columns:")
for col in feature_cols:
    print(f"- {col}")


Feature Columns:
- sales_last_7_days
- sales_last_30_days
- days_since_first_sale
- days_since_last_sale
- avg_price
- min_price
- max_price
- product_price_std
- customer_price_std
- num_purchases
- total_spent
- days_since_last_purchase
- avg_transaction_value
- avg_days_between_purchases
- primary_department
- primary_garment_group
- category_diversity


### Categorical features
One hot encode the garment groups. Drop primary department, too many possibilities

In [ ]:
print(f"Unique departments: {train_data['primary_department'].nunique()}")
print(f"Unique garment groups: {train_data['primary_garment_group'].nunique()}")

Unique departments: 56
Unique garment groups: 19


In [ ]:
train_data = train_data.drop('primary_department', axis=1)
val_data = val_data.drop('primary_department', axis=1)
test_data = test_data.drop('primary_department', axis=1)

train_data = pd.get_dummies(train_data, columns=['primary_garment_group'], prefix='garment')
val_data = pd.get_dummies(val_data, columns=['primary_garment_group'], prefix='garment')
test_data = pd.get_dummies(test_data, columns=['primary_garment_group'], prefix='garment')


all_columns = set(train_data.columns).union(val_data.columns).union(test_data.columns)
print(all_columns)

train_data = train_data.reindex(columns=all_columns, fill_value=0)
val_data = val_data.reindex(columns=all_columns, fill_value=0)
test_data = test_data.reindex(columns=all_columns, fill_value=0)

{'garment_Dressed', 'garment_Jersey Basic', 'garment_Dresses/Skirts girls', 'garment_Shoes', 'total_spent', 'min_price', 'sales_last_7_days', 'purchased', 'garment_Under-, Nightwear', 'days_since_first_sale', 'days_since_last_sale', 'customer_id', 'num_purchases', 'garment_Jersey Fancy', 'sales_last_30_days', 'garment_Shorts', 'garment_Woven/Jersey/Knitted mix Baby', 'article_id', 'garment_Unknown', 'days_since_last_purchase', 'garment_Outdoor', 'avg_transaction_value', 'garment_Trousers Denim', 'garment_Dresses Ladies', 'customer_price_std', 'garment_Trousers', 'garment_Blouses', 'garment_Accessories', 'max_price', 'avg_days_between_purchases', 'garment_Skirts', 'garment_Swimwear', 'avg_price', 'garment_Socks and Tights', 'garment_Knitwear', 'category_diversity', 'product_price_std', 'garment_Special Offers', 'garment_Shirts'}


### X, y split
Outcome variable is `purchased` and 0 or 1 feature

In [ ]:
y_train = train_data['purchased']
X_train = train_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_train = train_data['purchased']
print(f"{X_train.shape=}")
print(f"{y_train.shape=}")
X_val = val_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_val = val_data['purchased']
print(f"{X_val.shape=}")
print(f"{y_val.shape=}")
X_test = test_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_test = test_data['purchased']
print(f"{X_test.shape=}")
print(f"{y_test.shape=}")

X_train.shape=(1110, 36)
y_train.shape=(1110,)
X_val.shape=(1260, 36)
y_val.shape=(1260,)
X_test.shape=(1158, 36)
y_test.shape=(1158,)


## Model building

Build a neural net model with variable number of hidden layers.

- num_features: number of features in dataset
- hidden_layer_sizes: variable array of nodes in hidden layers of neural net
- learning_rate: hyperparameter learning rate
- activation: activation function used within hidden layers
- optimizer: Adam or SGD optimizer

Uses sigmoid activation for the final layer. Reports accuracy.

In [ ]:
def build_recommendation_model(num_features,
                                hidden_layer_sizes=[512],
                                learning_rate=0.001,
                                activation='relu',
                                optimizer="Adam"):
    tf.keras.backend.clear_session()
    tf.random.set_seed(67)

    model = tf.keras.Sequential()
    for i, layer_units in enumerate(hidden_layer_sizes):
        if i == 0:
            layer = tf.keras.layers.Dense(
                units=layer_units,
                input_shape=(num_features,),
                use_bias=True,
                activation=activation,
                kernel_initializer=tf.initializers.RandomNormal(stddev=0.01),
                bias_initializer=tf.initializers.RandomNormal(stddev=0.01)
            )
        else:
            layer = tf.keras.layers.Dense(
                units=layer_units,
                use_bias=True,
                activation=activation,
                kernel_initializer=tf.initializers.RandomNormal(stddev=0.01),
                bias_initializer=tf.initializers.RandomNormal(stddev=0.01)
            )
        model.add(layer)
    
    model.add(tf.keras.layers.Dense(
        units=1,
        use_bias=True,
        activation='sigmoid',
        kernel_initializer=tf.initializers.RandomNormal(stddev=0.01),
        bias_initializer=tf.initializers.RandomNormal(stddev=0.01)
        )
    )

    if optimizer == "SGD":
        keras_optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    elif optimizer == "Adam":
        keras_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    else:
        keras_optimizer = optimizer

    model.compile(
        loss=tf.keras.losses.BinaryCrossentropy(),
        optimizer=keras_optimizer,
        metrics=['accuracy']
    )
    
    return model

In [ ]:
model = build_recommendation_model(X_train.shape[1],
                                        hidden_layer_sizes=[512, 512],
                                        learning_rate=0.01,
                                        activation='relu',
                                        optimizer="Adam")
model.summary()
history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=20,
                    batch_size=256,
                    verbose=1)


/opt/anaconda3/envs/mids_env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-02-20 23:23:17.150715: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2026-02-20 23:23:17.150737: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-02-20 23:23:17.150743: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
2026-02-20 23:23:17.150762: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-02-20 23:23:17.150771: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] C

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 282,113 (1.08 MB)

 Trainable params: 282,113 (1.08 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20


2026-02-20 23:23:17.421991: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 74ms/step - accuracy: 0.6180 - loss: 37.8024 - val_accuracy: 0.8333 - val_loss: 50.2646
Epoch 2/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8333 - loss: 80.7092 - val_accuracy: 0.1667 - val_loss: 60.5766
Epoch 3/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6748 - loss: 65.5428 - val_accuracy: 0.8333 - val_loss: 26.7800
Epoch 4/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6874 - loss: 88.4637 - val_accuracy: 0.8333 - val_loss: 75.8650
Epoch 5/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6748 - loss: 82.6635 - val_accuracy: 0.8333 - val_loss: 146.1227
Epoch 6/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7721 - loss: 123.3192 - val_accuracy: 0.8333 - val_loss: 52.2675
Epoch 7/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8333 - loss: 93.9101 - val_accuracy: 0.8238 - val_loss: 6.1454
Epoch 8/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6829 - loss: 100.9652 - val_accuracy: 0.8333 - val_loss: 5

## Test the model